# Zeitreihenanalyse

## Raster importieren

Die im Script ***2_STAC_Suche_bis_Composites*** erstellten und als **`GeoTIFF`** abgespeicherten Raster werden nun zur weiteren Verarbeitung geladen. Grundsätzlich könnten die Raster einzeln eingelesen werden. Um jedoch einen automatisierten Workflow zu ermöglichen, werden mehrere Raster gleichzeitig geladen. Hierfür wird das Modul [glob](https://docs.python.org/3/library/glob.html) (zum Auffinden von Dateien mit ähnlichen Namen) in Kombination mit [rioxarray](https://corteva.github.io/rioxarray/html/rioxarray.html) (zum Laden der Rasterdaten) verwendet.

In [ ]:
import rioxarray as rxr

ndvi_2021 = rxr.open_rasterio("./Composites_GeoTIFF/NDVI_2021.tif").squeeze()

# .squeeze() entfernt die "unnötige" Banddimension --> dann habe ich nur noch x & y

In [ ]:
print(ndvi_2021)

In [ ]:
# mehrere Raster mit ähnlicher Bezeichnung suchen (mit glob)

import glob

files_ndvi = sorted(glob.glob("./Composites_GeoTIFF/NDVI_*.tif"))
files_nbr = sorted(glob.glob("./Composites_GeoTIFF/NBR_*.tif"))
files_rgb = sorted(glob.glob("./Composites_GeoTIFF/RGB_*.tif"))
files_mask = sorted(glob.glob("./Composites_GeoTIFF/ValidMask_*.tif"))

In [ ]:
files_ndvi

In [ ]:
# Raster laden mit rioxarray

rasters_ndvi = [rxr.open_rasterio(f).squeeze() for f in files_ndvi]
rasters_nbr = [rxr.open_rasterio(f).squeeze() for f in files_nbr]
rasters_rgb = [rxr.open_rasterio(f) for f in files_rgb] # bei RGB brauche ich die Banddimension --> daher nicht .squeeze()
rasters_mask = [rxr.open_rasterio(f).squeeze() for f in files_mask]

In [ ]:
# kurz überprüfen anhand eines Rasters

raster_test = rasters_ndvi[0] 

In [ ]:
print(raster_test)

In [ ]:
print(raster_test.rio.crs)

In [ ]:
print(raster_test.shape)

In [ ]:
print(raster_test.rio.nodata)

In [ ]:
# Wertebereich checken --> sollte zwischen -1 und +1 liegen.

print(float(raster_test.min()), float(raster_test.max()))

In [ ]:
# Prüfen, ob alle Raster gleich gross sind

for r in rasters_ndvi:
    print(r.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# plotten mit Rot --> Grün Farbskala & Penzentil-Stretch (für besseren Kontrast)
raster_test.plot(
    cmap = "RdYlGn",  
    figsize=(8, 8),
    add_colorbar=False,                                
    vmin=np.nanpercentile(raster_test.compute(), 2),     # setzt unteren Farbwert: der Wert, unter dem (hier) 2% der Daten liegen
    vmax=np.nanpercentile(raster_test.compute(), 98)     # setzt oberen Farbwert: der Wert, unter dem (hier) 98% der Daten liegen
)
plt.title("NDVI Testplot")
plt.axis("off")
plt.show()

## Farbtreue RGB-Composites erstellen

Zur Kontrolle des Tresholds für *"nicht-Vegetation"* sowie zur späteren Darstellung der ***Change Detection*** werden farbgetreue RGB-Composites des ersten und des letzten Jahres (2021 und 2025) erstellt. 

Mit dem Setzen eines Tresholds sollen auf dem Raster des ersten Jahres alle Pixel ausgeschlossen werden, die nicht Vegetation darstellen. Diese Maske wird auf die anderen Raster übertragen.

**Vorgehen:** Die Normalisierung erfolgt für beide Jahre gleich. Die Aufhellung und die Gamma-Korrektur muss manuell/iterativ angepasst, und mittels Plot überprüft werden.

- `alpha`: Farben intensivieren --> höheres Alpha = höherer Kontrast
- `beta`: Bild aufhellen --> höheres Beta = helleres Bild
- `gamma`: Kontrast anpassen --> höheres Gamma = weniger Kontrast / realistischere Farben

In [ ]:
rgb_2021 = rxr.open_rasterio("./Composites_GeoTIFF/RGB_2021.tif")
rgb_2025 = rxr.open_rasterio("./Composites_GeoTIFF/RGB_2025.tif")

In [ ]:
type(rgb_2021)

In [ ]:
print(rgb_2021.rio.crs)

In [ ]:
rgb_2021.shape

In [ ]:
# Bänder extrahieren

red_21 = rgb_2021.isel(band=0)
green_21 = rgb_2021.isel(band=1)
blue_21 = rgb_2021.isel(band=2)

red_25 = rgb_2025.isel(band=0)
green_25 = rgb_2025.isel(band=1)
blue_25 = rgb_2025.isel(band=2)

In [ ]:
# Aufhellung einstellen

def brighten(band, alpha, beta):
    return np.clip(alpha * band + beta, 0, 255)

alpha_21 = 0.19
alpha_25 = 0.12

beta_21 = 0.1
beta_25 = 0

# 2021
red_21_b = brighten(red_21, alpha_21, beta_21)
green_21_b = brighten(green_21, alpha_21, beta_21)
blue_21_b = brighten(blue_21, alpha_21, beta_21)

# 2025
red_25_b = brighten(red_25, alpha_25, beta_25)
green_25_b = brighten(green_25, alpha_25, beta_25)
blue_25_b = brighten(blue_25, alpha_25, beta_25)

In [ ]:
# Werte normalisieren auf 0-1 

def normalize(band):
    band_min = band.min()
    band_max = band.max()
    return (band - band_min) / (band_max - band_min)

red_21_n = normalize(red_21_b)
green_21_n = normalize(green_21_b)
blue_21_n = normalize(blue_21_b)

red_25_n = normalize(red_25_b)
green_25_n = normalize(green_25_b)
blue_25_n = normalize(blue_25_b)

In [ ]:
# Gamma-Korrektur

def gammacorr(band, gamma):
    return np.power(band, 1/gamma)

gamma_21 = 1.0
gamma_25 = 0.8

red_21_g = gammacorr(red_21_n, gamma_21)
green_21_g = gammacorr(green_21_n, gamma_21)
blue_21_g = gammacorr(blue_21_n, gamma_21)

red_25_g = gammacorr(red_25_n, gamma_25)
green_25_g = gammacorr(green_25_n, gamma_25)
blue_25_g = gammacorr(blue_25_n, gamma_25)

In [ ]:
# RGB zusammenbauen

rgb_img_21 = np.dstack((red_21_g, green_21_g, blue_21_g))
rgb_img_25 = np.dstack((red_25_g, green_25_g, blue_25_g))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(rgb_img_21)
axes[0].set_title("RGB 2021")
axes[0].axis("off")

axes[1].imshow(rgb_img_25)
axes[1].set_title("RGB 2025")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("Composites_Plots/RGB_21_25.png", dpi=300, bbox_inches="tight") 
plt.show()

In [ ]:
# Plots noch separat speichern

plt.figure(figsize=(8, 8))
plt.imshow(rgb_img_21)
plt.title("RGB 2021")
plt.axis("off")

plt.savefig("Composites_Plots/RGB_2021.png", dpi=300, bbox_inches="tight")
plt.close()

In [ ]:
plt.figure(figsize=(8, 8))
plt.imshow(rgb_img_25)
plt.title("RGB 2025")
plt.axis("off")

plt.savefig("Composites_Plots/RGB_2025.png", dpi=300, bbox_inches="tight")
plt.close()

## Ausschluss von Nicht-Vegetationspixel

Anhand eines Tresholds des NDVI aus dem ersten Jahr (2021) soll klassifiert werden, ob ein Pixel Vegetation darstellt oder nicht. Dazu wird das NDVI-Composite aus 2021 als erstes interaktiv dargestellt, um den Treshold abschätzen zu können. Anschliessend wird eine Maske mit dem Treshold erstellt, und die Nicht-Vegetationspixel werden extrahiert. 

Durch Überlagerung der "Nicht-Vegetationspixel" mit dem RGB-Composite kann der Treshold überprüft werden.

In [ ]:
import xarray as xr

ndvi_2021 = rasters_ndvi[0] 

In [ ]:
# Wertebereich checken --> sollte zwischen -1 und +1 liegen.

print(float(ndvi_2021.min()), float(ndvi_2021.max()))

In [ ]:
import hvplot.xarray

In [ ]:
ndvi_2021.hvplot(
    x="x",
    y="y",
    cmap="RdYlGn",
    width=500,
    height=500,
    title="NDVI 2021"
).opts(
    xaxis=None,
    yaxis=None,
    colorbar=False
)

In [ ]:
# Treshold zur Unterscheidung von Vegetation und Nicht-Vegetation anwenden

treshold = 0.3

ndvi_vegetation = xr.where(ndvi_2021 > treshold, 1, 0)

In [ ]:
# kurz plotten

ndvi_vegetation.plot.imshow()

In [ ]:
# Maske erstellen mit den Pixeln, die nicht Vegetation sind (=0)

valid = ndvi_2021.notnull() # gültige Pixel definieren
non_veg = xr.where((ndvi_2021 <= treshold) & valid, 1, np.nan) # Nicht-Vegetation nur innerhalb AOI

In [ ]:
# Maske

mask = non_veg.values == 1

# aufbereitetes RGB verwenden

rgb_visual_21 = rgb_img_21.copy()
rgb_overlay_21 = rgb_img_21.copy()

In [ ]:
# Nicht-Veg rot färben

rgb_overlay_21[mask] = [1, 0, 0]

# von 0–1 auf 0–255 bringen

rgb_visual_21_uint8 = (rgb_visual_21 * 255).astype("uint8")
rgb_overlay_21_uint8 = (rgb_overlay_21 * 255).astype("uint8")

In [ ]:
rgb_visual_21_uint8.shape

In [ ]:
# von (y, x, band) zu (band, y, x)
rgb_visual_21_uint8 = np.transpose(rgb_visual_21_uint8, (2, 0, 1))
rgb_overlay_21_uint8 = np.transpose(rgb_overlay_21_uint8, (2, 0, 1))

In [ ]:
rgb_visual_21_uint8.shape

In [ ]:
# Als GeoTIFF speichern

rgb_visual_xr = xr.DataArray(
    rgb_visual_21_uint8,
    dims=("band", "y", "x"),
    coords={
        "band": [1, 2, 3],
        "y": rgb_2021.y,
        "x": rgb_2021.x
    }
)

rgb_overlay_xr = xr.DataArray(
    rgb_overlay_21_uint8,
    dims=("band", "y", "x"),
    coords={
        "band": [1, 2, 3],
        "y": rgb_2021.y,
        "x": rgb_2021.x
    }
)

rgb_visual_xr = rgb_visual_xr.rio.write_crs(rgb_2021.rio.crs)
rgb_overlay_xr = rgb_overlay_xr.rio.write_crs(rgb_2021.rio.crs)

rgb_visual_xr.rio.to_raster("Composites_GeoTIFF/RGB_2021_visual.tif")
rgb_overlay_xr.rio.to_raster("Composites_GeoTIFF/RGB_2021_visual_overlay.tif")

In [ ]:
import leafmap.foliumap as leafmap

# Karte erstellen
m = leafmap.Map()

# Split Map mit Schieber
m.split_map(
    left_layer="Composites_GeoTIFF/RGB_2021_visual.tif",
    right_layer="Composites_GeoTIFF/RGB_2021_visual_overlay.tif",
    left_label="RGB 2021",
    right_label="Non-Vegetation Overlay 2021",
)

# Karte anzeigen
m

## Stack erstellen mit `xarray`

Mit dem Modul `os.path.basename()` von [os](https://docs.python.org/3/library/os.path.html#module-os.path) kann aus dem Dateipfad der Dateiname extrahiert werden. Daraus wird die Jahreszahl ausgelesen und als Integer gespeichert

Mit der Funktion `xr.concat()` werden die einzelnen xarray-Objekte entlang einer neu erzeugten Zeitdimension (*time*) zu einem gemeinsamen Stack zusammengeführt.

In [ ]:
# Jahre mit glob extrahieren für automatisierten Ablauf

import os

years = [
    int(os.path.basename(f)[5:9]) # Jahreszahl bei files_ndvi ist an Stelle 6:10 --> da Python bei 0 anfängt = 5:9
    for f in files_ndvi
]

years

In [ ]:
# Stack der Raster erstellen mit xarray

import xarray as xr

ndvi_stack = xr.concat(rasters_ndvi, dim="time")
ndvi_stack["time"] = years

nbr_stack = xr.concat(rasters_nbr, dim="time")
nbr_stack["time"] = years

In [ ]:
print(ndvi_stack)

In [ ]:
type(ndvi_stack)

In [ ]:
print(ndvi_stack.dims)

## Differenzkarten erstellen

Jahr anpassen um andere Differenzen zu rechnen!

Interpretation:
- negative Werte = Vegetationsverlust
- positive Werte = Vegetationszunahme

In [ ]:
year_before = 2021
year_after = 2025

ndvi_before = ndvi_stack.sel(time=year_before)
ndvi_after = ndvi_stack.sel(time=year_after)

nbr_before = nbr_stack.sel(time=year_before)
nbr_after = nbr_stack.sel(time=year_after)

In [ ]:
dndvi = ndvi_after - ndvi_before
dnbr = nbr_after - nbr_before

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

dnbr.plot.imshow(
    ax=ax,
    cmap="RdYlGn",
    vmin=-0.6,
    vmax=0.6,
    add_colorbar=True
)

ax.set_title(f"ΔNBR | {year_after} - {year_before}")
ax.set_axis_off()

plt.show()

In [ ]:
# Tresholds für Abholzung definieren

treshold = -0.4

deforestation_ndvi = dndvi < treshold
deforestation_nbr = dnbr < treshold

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

deforestation_ndvi.plot.imshow(
    ax=ax,
    add_colorbar=False
)

ax.set_title(f"Potenzielle Abholzung NDVI | {year_before}–{year_after}")
ax.set_axis_off()

plt.show()

In [ ]:
dndvi_25_21 = ndvi_2025 - ndvi_2021

In [ ]:
dndvi_25_21.plot(
    cmap = "RdYlGn",  
    figsize=(8, 8),                           
)
plt.title("NDVI 2025 - 2021")
plt.axis("off")
plt.show()

### Kombinierte Betrachtung: NDVI & NBR < -0.xx

In [ ]:
treshold = -0.3

deforestation = (dndvi < treshold) & (dnbr < treshold)